# Steam 简体中文评测 · 探索分析

数据来源：`data/database/steam_reviews.db`（由 `scripts/collect.py` 采集、`scripts/import_data.py` 入库）。

口径固定为：5 款游戏、北京时间 2026-08-23 ~ 2026-11-21、简体中文、全部推荐状态、全部购买来源。

本 notebook 只做探索；稳定逻辑应回归 `src/steam_reviews/analysis/`。

In [ ]:
import sqlite3
from pathlib import Path

DB = Path('..') / 'data' / 'database' / 'steam_reviews.db'
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row

def q(sql, *params):
    return [dict(r) for r in conn.execute(sql, params)]

q('SELECT COUNT(*) AS total, MIN(created_at) AS first, MAX(created_at) AS last FROM reviews')

## 1. 各游戏采集量与本窗口好评率

推荐比例分母只包含 `is_recommended` 非 NULL 的记录。

In [ ]:
rows = q('''
SELECT appid,
       COUNT(*) AS total,
       SUM(CASE WHEN is_recommended = 1 THEN 1 ELSE 0 END) AS positive,
       SUM(CASE WHEN is_recommended = 0 THEN 1 ELSE 0 END) AS negative,
       SUM(CASE WHEN is_recommended IS NULL THEN 1 ELSE 0 END) AS missing
FROM reviews GROUP BY appid ORDER BY total DESC
''')
for r in rows:
    known = r['positive'] + r['negative']
    r['rate'] = round(r['positive'] / known, 4) if known else None
    print(r)

## 2. 采集完整性

每段状态、页数与去重结果；`completed` 表示以空评测列表正常结束。

In [ ]:
for r in q('''
SELECT run_id, appid, segment_index, status, pages, fetched, inserted, updated, errors
FROM crawl_segments ORDER BY run_id, appid, segment_index
'''):
    print(r)

q('SELECT COUNT(*) AS distinct_ids FROM (SELECT DISTINCT review_id FROM reviews)')

## 3. 发表时游玩时长分组

`playtime_at_review` 是发表评测那一刻的时长，不是当前累计时长。

In [ ]:
buckets = [(0,60,'0-1h'),(60,300,'1-5h'),(300,1200,'5-20h'),
           (1200,3000,'20-50h'),(3000,6000,'50-100h'),(6000,None,'100h+')]
for lo, hi, label in buckets:
    sql = '''SELECT COUNT(*) n,
                    SUM(CASE WHEN is_recommended=1 THEN 1 ELSE 0 END) p,
                    SUM(CASE WHEN is_recommended=0 THEN 1 ELSE 0 END) ng
             FROM reviews WHERE playtime_at_review >= ?'''
    params = [lo*60]
    if hi is not None:
        sql += ' AND playtime_at_review < ?'
        params.append(hi*60)
    row = q(sql, *params)[0]
    known = (row['p'] or 0) + (row['ng'] or 0)
    rate = round((row['p'] or 0)/known, 4) if known else None
    print(f"{label:<9} n={row['n']:<6} rate={rate}")

## 4. 按日评论量与好评率（北京时间）

In [ ]:
daily = q('''
SELECT strftime('%Y-%m-%d', created_at, 'unixepoch', '+8 hours') AS day,
       COUNT(*) n,
       SUM(CASE WHEN is_recommended=1 THEN 1 ELSE 0 END) p,
       SUM(CASE WHEN is_recommended=0 THEN 1 ELSE 0 END) ng
FROM reviews GROUP BY day ORDER BY day
''')
print('days:', len(daily), '| peak:', max(daily, key=lambda r: r['n']) if daily else None)
daily[:5]

## 5. 缺失率

未知一律 NULL，不用 0 或空串替代；缺失率高说明该字段不宜用于结论。

In [ ]:
fields = ['author_steamid','content','is_recommended','playtime_at_review',
          'refunded','developer_response','cpu_name','vram_size']
total = q('SELECT COUNT(*) n FROM reviews')[0]['n']
for f in fields:
    miss = q(f'SELECT SUM(CASE WHEN {f} IS NULL THEN 1 ELSE 0 END) m FROM reviews')[0]['m'] or 0
    print(f'{f:<24} missing={miss:<8} rate={miss/total:.2%}' if total else f)